<div align="left">

# 第 4 册：完整版工作流——从快跑到全管控

零基础篇你用 Flash 轻量版交付了 add_custom 和 sigmoid_custom——11 分钟，一个 Agent，自走八阶段。本册不再复述流程细节，它回答三个更根本的问题：**什么时候值得为治理付出时间成本？完整版为什么这样设计？你在其中扮演什么角色？**

本册附一段真实运行（Ascend 910B3 + OpenCode + CANN 9.1，全程 **2.5 小时**）的精编证据。

路线：**为什么两套 → 三大设计思想 → 你要做的三类事 → 实录精编 → 选型总结**。全程约 40 分钟；课后实践另计。

> **环境提示**：想复现实录，进入 cannbot-skills 的 `plugins-official/ops-direct-invoke`，用 `bash init.sh project opencode ~/adv-operators` 初始化完整版插件（比 Flash 版重：45 Skill / 6 Agent / 3 依赖仓）。
</div>

<div align="left">

## 4.1 为什么需要两套工作流

一句话原理：**治理强度应该随风险缩放**。Flash 版像自测代码——快，但出了问题只有你自己知道；完整版像带评审的正式交付——慢，但每一步有独立验收、有证据可查。为原型付治理成本是浪费，为正式交付省治理成本是冒险。

什么信号告诉你该切完整版：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信号</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">为什么</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">要交付给客户 / 进生产仓</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">错误的代价不再只由你承担</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">有精度指标或审计要求</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">需要可追溯的验收证据链</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子复杂（条件语义 / Cast 链 / 规约）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">实现出错概率高，需要独立评审拦截</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">首次与 Agent 协作开发</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">边界感（哪些放权、哪些拍板）尚未建立</td></tr></tbody></table>

## 4.2 完整版的三大设计思想

<img src="./images/04_complete_workflow.png" alt="完整版工作流：八阶段×CP门禁×多Agent团队" width="800px">

**思想一：分权制衡——把原则变成机制。** 零基础篇第 1 册说过「写代码的不能自己宣布通过」；完整版把它落成了组织结构：PM 只调度不干活，developer-code/test/doc 各司其职，**QA 用独立的 workflow-cp* Skill 验收六道 CP 门禁（CP0~CP5），不通过就打回**。你不再需要「相信」Agent 自检——验收权在第三方手里。

**思想二：决策权恰好留给人的三个点。** 用户问卷不在每个阶段都发，只出现在 **CP0（环境）/ CP1（需求）/ CP2.2（方案）**——因为这三处是**语义性与不可逆决策点**：环境错了全盘皆错，需求是误差的源头，方案一旦定向下游全按它执行。其余阶段是执行细节，**放权给 Agent 换取无人值守的推进速度**。这是「人在哪介入」的通用设计模式：**拍板语义，放权执行**。

**思想三：过程与产物分离。** 流程产物（state.json 台账、问卷、验收报告）落 `.cannbot/<算子名>/`，交付物（代码/测试/文档）落工作区专业目录。过程可审计、产物可提交——本次实录留下 19 个流程节点、6 份验收报告、8 项排障记录，全部可回溯。
</div>

<div align="left">

## 4.3 作为决策者驱动它：你要做的三类事

完整版里你的角色从「观察者」变成「决策者」：

**① 写一份含规则顺序的需求。** 实录的需求是 xlog1py：`z = x * log1p(y)`，两条规则且**顺序写死**——「x==0 → z=0 先应用；y 为 NaN → z=y 后应用并覆盖」。为什么要写到这个粒度？因为 x=0 且 y=NaN 时，输出是 NaN 还是 0，两条规则的应用顺序不同答案就不同——**规则顺序是语义的一部分**，不写清楚 Agent 就得猜。零基础篇的五要素在这里升级为「要素 + 语义优先级」。

**② 在三处问卷拍板。** 实录中 CP1 的四个拍板：dtype 范围（仅 fp32）、输入约定（等形，host 物化广播）、测试规模、架构选型（SIMD）。每个拍板都是一行决策——记住「等形输入」这个当时省事的决定，**第 5 册的性能数据会重新审视它的代价**。

**③ 读门禁报告再放行。** 每个 CP 的验收报告是放行依据；被 QA 打回时，**按打回原因回退到对应阶段修复后重走**（功能问题回 3.1，精度口径问题回 2.1；回退路径记录在 state.json）——不是跳过门禁，也不是弃用完整版换 Flash。
</div>

<div align="left">

## 4.4 实录精编：一次 2.5 小时的完整运行

> 完整产物在实录工程 `~/adv-operators`（.cannbot/ 全套文档 + direct-launch 提交工程）。以下只保留**改变认知的节点**。

**门禁的价值（CP2.2 转折）。** 朴素实现 `Ln(1+y)` 有个精度陷阱：y 极小时 `1+y` 的舍入会吞掉有效位（fp32 下 1+1e-8 就是 1.0）。方案采用恒等式 `log1p(y) = log(1+y) · y/((1+y)−1)`——分子分母的舍入误差相消，配两条回退。**QA 在 CP2.2 做独立逐点推导时，发现 y=+Inf 时 Inf/Inf=NaN 会污染结果**——开发方案据此补上第三条回退。这就是门禁的意义：缺陷在写代码之前被数学推导拦下，而不是在真机上被精度失败暴露。

**联调排障：两条最有迁移性的。** 8 项排障记录在案，最值得带走两条：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">教训</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">为什么重要</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">静默截断比报错更危险</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Compare 系 API 在 910b 上要求 count×4B 为 256B 整数倍，不满足时不报错、直接截断——专项边界用例是唯一防线</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立参考防的不只是 Agent</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">冒烟用例的手算期望值算错了，是 golden 比对把「人」的错误拦下来——手算与 golden 双通道缺一不可</td></tr></tbody></table>

**关键数字**：全程 2.5 小时（Flash 版同量级 11 分钟）；**47/47** 精度通过（rtol=1e-5/atol=1e-7 对 float64 golden，含 NaN/±Inf/y=-1/极小 y 共 15 组边界全对）；产物为可上库的 **direct-launch 四件套**（kernel/launch/plugin/CMakeLists；内核逻辑一句话：14 条向量指令的三级流水 + 双缓冲，细节见工程）。
</div>

<div align="left">

## 4.5 两版对比与「学完怎么用」

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"></th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Flash 版（零基础篇·册 2）</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">完整版（本册）</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">耗时</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">~11 分钟</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">~2.5 小时</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">评审</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">simple 档自动跳过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">CP2.2 数学逐点复核，**拦下 Inf 回退缺陷**</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">排障</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">0 项（参考算子照抄）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">8 项（含 golden 拦下人的手算错误）</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">交付物</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">.asc 单文件 + pytest</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">**可上库的提交工程四件套** + 全套验收报告</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">适用</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">原型 / 学习 / simple 档</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">正式交付 / 复杂算子 / 要质量证据</td></tr></tbody></table>

2.5 小时买到了什么——**可追溯的质量证据链**与**可上库的工程形态**。当你交付的算子要进生产仓库、要对他人负责时，这笔投资是值得的。

**行动清单（下次开发前问自己）**：

- 交付给谁？错误代价谁承担？→ 决定用哪套工作流；
- 需求里有没有「规则顺序」级别的语义？→ 写死它；
- 问卷来了，你是在拍板还是在走过场？→ 每个拍板都问一句「这个决定的代价什么时候回来找我」。

## 4.6 课后练习

> 💡 **进阶篇实践题是一条连续的项目线**：第 4 册开发你的算子 → 第 5 册为它做性能调优 → 第 6 册把它提 PR 上库。建议三册沿用**同一个算子**。

1. （单选题）你要把一个带条件语义的算子交付进生产仓，有精度指标与审计要求。应该？  
    A. 用 Flash 版快速交付，反正有 pytest  
    B. 用完整版工作流——治理成本换来可追溯的验收证据链  
    C. 先 Flash 后完整版各做一遍，双保险  
    D. 完整版太慢，手动开发最可靠

2. （单选题）用户问卷为什么恰好设在 CP0（环境）/ CP1（需求）/ CP2.2（方案），而不是每个阶段都问？  
    A. 这三个阶段的 Agent 能力最弱  
    B. 这三处是语义性与不可逆决策点，错了代价最大；其余是执行细节，放权换取推进速度  
    C. 问卷太多用户会烦，随机选了三个  
    D. 历史遗留，没有设计原因

3. （单选题）你的算子在 CP3 功能验收被 QA 打回。正确的动作是？  
    A. 让 developer Agent 忽略门禁继续推进  
    B. 按打回原因回退到对应阶段修复，重走后续流程  
    C. 放弃完整版，改用 Flash 版绕过门禁  
    D. 手工修改测试用例让它通过

4. （编程实践·必做）用完整版工作流开发一个你选定的算子（建议从双输入 + 条件语义起步），走完 0→CP5 全流程。提交：`.cannbot/` 全套产物 + **决策日志**——你在哪些点拍了什么板、为什么、你预计哪个决定的代价会在后续回来（参考本册「等形输入」的伏笔写法）。

5. （编程实践·选做）体验静默模式：`bash init.sh project opencode --mode silent` 初始化静默工程（或在会话中对 PM 说「开启静默模式」），开发一个简单算子，对比两种模式的差异——重点观察：交互模式里你拍板的那些决策在静默模式下去了哪里（`.cannbot/settings.json` 的 mode 字段与问卷落盘差异）。提交：两种模式推进日志对比 + 一段「什么场景用哪种模式」的结论。

**执行以下代码获取答案。**
</div>

In [ ]:
!cat ./answer/04_answer.txt

<div align="left">

## 本册小结

- **选型**：治理强度随风险缩放——客户交付 / 审计 / 复杂算子 / 首次协作是切完整版的信号；
- **三大设计思想**：分权制衡（原则变机制）、决策权恰好留给人的三个点（拍板语义，放权执行）、过程与产物分离（可审计）；
- **你的角色**：写含规则顺序的需求 / 三处问卷真拍板 / 读门禁报告再放行；
- **证据**：2.5 小时实录；QA 在 CP2.2 拦下 Inf 缺陷（门禁在写代码之前拦缺陷）；47/47；golden 拦下人的手算错误（独立参考防的不只是 Agent）。

下一册：[性能调优——数据驱动的迭代闭环](./05_performance_optimization.ipynb)，给本册交付的 xlog1py 做体检与提速。
</div>